<a href="https://colab.research.google.com/github/prachissharma/DataAnalyst/blob/main/FLEET_ANALYSIS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --------------------------------------------
# FLEET ANALYTICS ML MODEL USING RANDOM FOREST
# --------------------------------------------

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score
import matplotlib.pyplot as plt

# --------------------------------------------
# STEP 1: LOAD DATA
# --------------------------------------------
# Your dataset should contain columns like:
# fuel_consumption, mileage, maintenance_cost, repair_count, driver_score, cost_per_trip, downtime_hours

df = pd.read_csv("fleet_data.csv")
print(df.head())

# --------------------------------------------
# STEP 2: HANDLE MISSING DATA
# --------------------------------------------
df = df.fillna(df.mean(numeric_only=True))

# --------------------------------------------
# STEP 3: FEATURE ENGINEERING
# --------------------------------------------
df["fuel_efficiency"] = df["mileage"] / df["fuel_consumption"]
df["repair_ratio"] = df["repair_count"] / (df["mileage"] + 1)
df["cost_efficiency"] = df["cost_per_trip"] / (df["fuel_consumption"] + 1)

# Predictive Target: Maintenance Cost (Regression)
X = df.drop(["maintenance_cost"], axis=1)
y = df["maintenance_cost"]

# --------------------------------------------
# STEP 4: TRAIN-TEST SPLIT
# --------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --------------------------------------------
# STEP 5: TRAIN RANDOM FOREST
# --------------------------------------------
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

# --------------------------------------------
# STEP 6: MODEL EVALUATION
# --------------------------------------------
y_pred = rf.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("R2 Score:", r2)

# --------------------------------------------
# STEP 7: FEATURE IMPORTANCE (KEY INSIGHTS)
# --------------------------------------------
importances = pd.Series(rf.feature_importances_, index=X.columns)
importances.sort_values().plot(kind='barh', figsize=(8,6))
plt.title("Key Factors Affecting Maintenance Cost")
plt.xlabel("Importance Score")
plt.show()

print("\nTop Influencing Factors:")
print(importances.sort_values(ascending=False))

# --------------------------------------------
# STEP 8: PREDICTION FUNCTION (AI MODEL FOR FLEET ANALYST)
# --------------------------------------------

def predict_maintenance_cost(fuel, mileage, repairs, driver_score, cost_trip, downtime):

    sample = pd.DataFrame({
        "fuel_consumption": [fuel],
        "mileage": [mileage],
        "repair_count": [repairs],
        "driver_score": [driver_score],
        "cost_per_trip": [cost_trip],
        "downtime_hours": [downtime],
        "fuel_efficiency": [mileage / (fuel + 1)],
        "repair_ratio": [repairs / (mileage + 1)],
        "cost_efficiency": [cost_trip / (fuel + 1)]
    })

    prediction = rf.predict(sample)[0]
    return prediction

# Example Prediction
print("\nPredicted Maintenance Cost:",
      predict_maintenance_cost(40, 500, 2, 85, 1200, 5))
